<a href="https://colab.research.google.com/github/martinhdezpacheco/tfg-scraping-madrid/blob/main/extraer_inmueble_tecnocasa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4

In [ ]:
'''================================================
FASE 1 (TECNOCASA) : Extracción de URLs de Tecnocasa
================================================'''

# ATENCIÓN!!!!!! Eliminar antes los archivos de Colab

import requests
from bs4 import BeautifulSoup
import json
import html
import time
import csv
import os

In [ ]:
"""EJEMPLO DE EXTRACCIÓN DE DATOS DE UNA FICHA DE INMUEBLE EN TECNOCASA"""

def extraer_datos_inmueble(url):
    """
    Descarga y extrae los datos de una única ficha de inmueble de Tecnocasa.

    Parámetro:
        url (str): URL completa de la ficha, ej:
                   "https://www.tecnocasa.es/venta/piso/madrid/madrid/664738.html"

    Devuelve:
        dict con los campos limpios, o None si algo falló.
    """

    # --- PASO 1: Descargar el HTML crudo ---
    # Añadimos un "User-Agent" que simula un navegador real. En
    # books.toscrape.com no hacía falta porque es una web de práctica
    # sin ninguna protección, pero en una web real como Tecnocasa,
    # si no lo mandamos, el servidor puede rechazar la petición o
    # devolver una versión distinta de la página (o directamente un error).
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        respuesta = requests.get(url, headers=headers, timeout=10)
        respuesta.raise_for_status()  # lanza un error si la petición falló (404, 500...)
    except requests.exceptions.RequestException as e:
        print(f"Error al descargar {url}: {e}")
        return None

    # --- PASO 2: Parsear el HTML con BeautifulSoup ---
    soup = BeautifulSoup(respuesta.text, "html.parser")

    # --- PASO 3: Encontrar la etiqueta <estate-show-v2> ---
    # Aunque no es una etiqueta HTML "de verdad" (es un componente Vue.js
    # personalizado), BeautifulSoup la trata exactamente igual que
    # cualquier otra etiqueta (<div>, <span>, etc.) porque para
    # BeautifulSoup, cualquier palabra entre < > es una etiqueta válida.
    tag_estate = soup.find("estate-show-v2")

    if tag_estate is None:
        print(f"No se encontró <estate-show-v2> en {url}")
        return None

    # --- PASO 4: Extraer el atributo ":estate" ---
    # Este atributo empieza por ":" (es la sintaxis de Vue.js para un
    # "binding dinámico"). Se accede igual que a cualquier otro atributo,
    # con .get("nombre_del_atributo").
    json_crudo = tag_estate.get(":estate")

    if json_crudo is None:
        print(f"No se encontró el atributo :estate en {url}")
        return None

    # --- PASO 5: "Desescapar" el texto ---
    # El HTML convierte las comillas dobles en &quot; para poder meter
    # un JSON entero dentro de un atributo HTML sin romper la sintaxis.
    # html.unescape() deshace esa conversión (&quot; -> ", &amp; -> &, etc.)
    json_texto = html.unescape(json_crudo)

    # --- PASO 6: Convertir el texto en un diccionario de Python ---
    try:
        datos_completos = json.loads(json_texto)
    except json.JSONDecodeError as e:
        print(f"Error al parsear JSON en {url}: {e}")
        return None

    # --- PASO 7: Quedarnos solo con los campos que nos interesan ---
    # datos_completos es un diccionario ENORME (incluye fotos, colegios
    # cercanos, farmacias, datos de la agencia, hipoteca...). Nosotros
    # solo necesitamos los campos relevantes para el dataset de precios.
    #
    # Usamos .get() en vez de [] porque .get() no da error si la clave
    # no existe (devuelve None) - útil porque no todas las fichas tienen
    # rellenos todos los campos.
    # Nota sobre "features.elevator" y campos similares (garden, concierge...):
    # cuando el JSON trae "" (cadena vacía) NO sabemos con certeza si significa
    # "no tiene" o "no se especificó" en el anuncio. Por seguridad, tratamos
    # cualquier valor vacío como dato AUSENTE (None), no como "no" - así evitamos
    # meter un sesgo falso en el modelo. Más adelante, con más fichas descargadas,
    # se puede revisar si esto se puede refinar comparando con el texto libre
    # de la descripción.
    inmueble = {
        "id": datos_completos.get("id"),
        "detail_url": datos_completos.get("detail_url"),
        "tipo": datos_completos.get("type", {}).get("title"),
        "distrito": datos_completos.get("district", {}).get("title"),
        "barrio": datos_completos.get("quarter"),
        "precio": datos_completos.get("numeric_price"),                       # ej: 1399000
        "m2": datos_completos.get("numeric_surface"),                         # ej: "160.00"
        "habitaciones": datos_completos.get("rooms"),                         # ej: "3 dorm." (a limpiar con regex)
        "banos": datos_completos.get("bathrooms"),                            # ej: "3 baños" (a limpiar con regex)
        "planta": datos_completos.get("features", {}).get("floor"),
        "anio_construccion": datos_completos.get("features", {}).get("build_year"),
        "anio_reforma": datos_completos.get("features", {}).get("renovation_year"),
        "categoria": datos_completos.get("features", {}).get("category"),
        "ascensor": datos_completos.get("features", {}).get("elevator"),
        "balcones": datos_completos.get("features", {}).get("balconies"),
        "terrazas": datos_completos.get("features", {}).get("terraces"),
        "jardin": datos_completos.get("features", {}).get("garden"),
        "calefaccion": datos_completos.get("features", {}).get("heating"),
        "clase_energetica": datos_completos.get("energy_data", {}).get("class"),
        # points_of_interest es una estructura anidada (colegios, farmacias,
        # bares... cada uno con nombre y distancia). No cabe en una sola
        # celda de forma "limpia", así que la guardamos como texto JSON
        # completo por ahora, para procesarla aparte más adelante.
        "points_of_interest": json.dumps(datos_completos.get("points_of_interest"), ensure_ascii=False),
        "title": datos_completos.get("title"),
        "description": datos_completos.get("description"),
    }

    # Normalizamos: cualquier campo que sea cadena vacía ("") lo convertimos
    # en None, para que al guardar en CSV quede en blanco de verdad, en vez
    # de aparecer como texto vacío o como la palabra "None".
    for clave, valor in inmueble.items():
        if valor == "":
            inmueble[clave] = None

    return inmueble


# --- PRUEBA: extraemos un único piso para comprobar que funciona ---
if __name__ == "__main__":
    url_prueba = "https://www.tecnocasa.es/venta/piso/madrid/madrid/664738.html"
    resultado = extraer_datos_inmueble(url_prueba)

    if resultado:
        print("Datos extraídos correctamente:\n")
        for campo, valor in resultado.items():
            print(f"{campo}: {valor}")
    else:
        print("No se pudieron extraer los datos.")

Datos extraídos correctamente:

id: 664738
detail_url: https://www.tecnocasa.es/venta/piso/madrid/madrid/664738.html
tipo: Piso
distrito: Las Letras Y Cortes
barrio: Huertas - Cortes
precio: 1399000
m2: 160.00
habitaciones: 3 dorm.
banos: 3 baños
planta: 3 (planta ático)
anio_construccion: 1880
anio_reforma: 2002
categoria: Media
ascensor: None
balcones: None
terrazas: None
jardin: None
calefaccion: centralizada (Radiadores)
clase_energetica: e
points_of_interest: {"public_transport": [{"name": "Antón Martín", "class": "railway", "subclass": "subway", "icon": "subway", "distance": "200 m"}, {"name": "Lavapiés", "class": "railway", "subclass": "subway", "icon": "subway", "distance": "550 m"}, {"name": "Sol", "class": "railway", "subclass": "station", "icon": "station", "distance": "770 m"}, {"name": "Madrid-Puerta de Atocha", "class": "railway", "subclass": "station", "icon": "station", "distance": "930 m"}, {"name": "Atocha - Costanilla Desamparados", "class": "bus", "subclass": "bus_s

In [ ]:
"""===================================================================
FASE 1 (TECNOCASA) BLOQUE 0: SUBIR EL CSV DEL DÍA ANTERIOR (si existe)
==================================================================="""

from google.colab import files

print("Si ya tienes un CSV de un día anterior, selecciónalo ahora. Si es la primera vez, pulsa 'Cancelar' o cierra el selector.")
subido = files.upload()

print("Bloque 0 completado.")

Si ya tienes un CSV de un día anterior, selecciónalo ahora. Si es la primera vez, pulsa 'Cancelar' o cierra el selector.


Saving URLs_recolectadas.csv to URLs_recolectadas.csv
Bloque 0 completado.


In [ ]:
"""===================================================================
FASE 1 (TECNOCASA) BLOQUE 1: EXTRACCIÓN DE TODAS LAS URLs (estado actual de Tecnocasa)
==================================================================="""

todas_las_urls = []
pagina = 1
ids_pagina_1 = set()

while True:
    if pagina == 1:
        url_listado = "https://www.tecnocasa.es/venta/inmuebles/comunidad-de-madrid/madrid/madrid.html"
    else:
        url_listado = f"https://www.tecnocasa.es/venta/inmuebles/comunidad-de-madrid/madrid/madrid.html/pag-{pagina}"

    respuesta = requests.get(url_listado, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)

    if respuesta.status_code != 200:
        print(f"Página {pagina} no disponible (status {respuesta.status_code}). Fin.")
        break

    soup = BeautifulSoup(respuesta.text, "html.parser")
    tag_estates = soup.find("estates-index")

    if tag_estates is None:
        print(f"No se encontró <estates-index> en la página {pagina}. Fin.")
        break

    json_texto = html.unescape(tag_estates.get(":estates"))
    lista_anuncios = json.loads(json_texto)

    if len(lista_anuncios) == 0:
        print(f"Página {pagina} sin anuncios. Fin.")
        break

    ids_actuales = {anuncio.get("id") for anuncio in lista_anuncios}

    if pagina == 1:
        ids_pagina_1 = ids_actuales
    else:
        coincidencias = len(ids_actuales & ids_pagina_1)
        if coincidencias >= len(ids_actuales) / 2:
            print(f"Página {pagina} parece ser fallback. Fin real del listado.")
            break

    for anuncio in lista_anuncios:
        url_detalle = anuncio.get("detail_url")
        if url_detalle:
            todas_las_urls.append(url_detalle)

    print(f"Página {pagina}: {len(lista_anuncios)} anuncios encontrados (acumulado: {len(todas_las_urls)})")

    pagina += 1
    time.sleep(1)

print(f"Bloque 1 completado. Total de URLs en Tecnocasa hoy: {len(todas_las_urls)}")

Página 1: 15 anuncios encontrados (acumulado: 15)
Página 2: 15 anuncios encontrados (acumulado: 30)
Página 3: 15 anuncios encontrados (acumulado: 45)
Página 4: 15 anuncios encontrados (acumulado: 60)
Página 5: 15 anuncios encontrados (acumulado: 75)
Página 6: 15 anuncios encontrados (acumulado: 90)
Página 7: 15 anuncios encontrados (acumulado: 105)
Página 8: 15 anuncios encontrados (acumulado: 120)
Página 9: 15 anuncios encontrados (acumulado: 135)
Página 10: 15 anuncios encontrados (acumulado: 150)
Página 11: 15 anuncios encontrados (acumulado: 165)
Página 12: 15 anuncios encontrados (acumulado: 180)
Página 13: 15 anuncios encontrados (acumulado: 195)
Página 14: 15 anuncios encontrados (acumulado: 210)
Página 15: 15 anuncios encontrados (acumulado: 225)
Página 16: 15 anuncios encontrados (acumulado: 240)
Página 17: 15 anuncios encontrados (acumulado: 255)
Página 18: 15 anuncios encontrados (acumulado: 270)
Página 19: 15 anuncios encontrados (acumulado: 285)
Página 20: 15 anuncios enco

In [ ]:
"""======================================================================
FASE 1 (TECNOCASA) BLOQUE 2: EXTRACCIÓN DE LAS URLs NUEVAS (comparando con el CSV existente)
======================================================================"""

nombre_csv = "URLs_recolectadas.csv"

urls_antiguas = set()
if os.path.exists(nombre_csv):
    with open(nombre_csv, "r", encoding="utf-8-sig") as archivo:
        lector = csv.DictReader(archivo)
        for fila in lector:
            urls_antiguas.add(fila["url_detalle"])

urls_nuevas = set(todas_las_urls) - urls_antiguas

print(f"Bloque 2 completado: {len(urls_antiguas)} URLs antiguas leídas, {len(urls_nuevas)} URLs nuevas detectadas.")

Bloque 2 completado: 982 URLs antiguas leídas, 10 URLs nuevas detectadas.


In [ ]:
"""===================================================================
FASE 1 (TECNOCASA) BLOQUE 3: CONTEO — ANTES vs. DESPUÉS DE LA ACTUALIZACIÓN
==================================================================="""

print(f"URLs que había antes de esta ejecución: {len(urls_antiguas)}")
print(f"URLs nuevas encontradas hoy: {len(urls_nuevas)}")
print(f"Total tras la actualización: {len(urls_antiguas) + len(urls_nuevas)}")
print("Bloque 3 completado.")

URLs que había antes de esta ejecución: 982
URLs nuevas encontradas hoy: 10
Total tras la actualización: 992
Bloque 3 completado.


In [ ]:
"""===================================================================
FASE 1 (TECNOCASA) BLOQUE 4: EXPORTAR — AÑADIR LAS URLs NUEVAS AL CSV
==================================================================="""

archivo_existe = os.path.exists(nombre_csv)

with open(nombre_csv, "a", newline="", encoding="utf-8-sig") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=["url_detalle"])
    if not archivo_existe:
        escritor.writeheader()
    for url in urls_nuevas:
        escritor.writerow({"url_detalle": url})

print(f"Bloque 4 completado. Se han añadido {len(urls_nuevas)} URLs nuevas a {nombre_csv}")

Bloque 4 completado. Se han añadido 10 URLs nuevas a URLs_recolectadas.csv


In [ ]:
"""===================================================================
FASE 1 (TECNOCASA) BLOQUE 5: DESCARGAR EL CSV ACTUALIZADO
==================================================================="""

files.download(nombre_csv)

print("Bloque 5 completado. Guarda este CSV para subirlo mañana en el Bloque 0.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Bloque 5 completado. Guarda este CSV para subirlo mañana en el Bloque 0.


In [ ]:
'''================================================
FASE 2 (TECNOCASA): Extracción de datos de cada URL de Tecnocasa
================================================'''

# ATENCIÓN!!!!!! Eliminar antes los archivos de Colab

import requests
from bs4 import BeautifulSoup
import json
import html
import time
import csv
import os
from google.colab import files

def extraer_datos_inmueble(url):
    try:
        respuesta = requests.get(url, timeout=10)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException:
        return None

    soup = BeautifulSoup(respuesta.text, "html.parser")
    etiqueta = soup.find("estate-show-v2")

    if etiqueta is None:
        return None

    estate_raw = etiqueta.get(":estate")
    if estate_raw is None:
        return None

    estate_json = html.unescape(estate_raw)
    estate = json.loads(estate_json)

    # Sub-objetos anidados donde viven planta, año, ascensor, etc.
    features = estate.get("features") or {}
    energy_data = estate.get("energy_data") or {}

    # "tipo" y "distrito" vienen como diccionarios {"id":..., "title":..., "slug":...}
    # nos quedamos solo con el texto legible ("title")
    tipo_dict = estate.get("type") or {}
    distrito_dict = estate.get("district") or {}

    # m2 viene como string numérico limpio ("62.00"), lo convertimos a float directamente
    m2_raw = estate.get("numeric_surface")
    try:
        m2_valor = float(m2_raw) if m2_raw is not None else None
    except ValueError:
        m2_valor = None

    datos = {
        "id": estate.get("id"),
        "detail_url": url,
        "tipo": tipo_dict.get("title"),
        "distrito": distrito_dict.get("title"),
        "barrio": estate.get("quarter"),
        "precio": estate.get("numeric_price"),
        "m2": m2_valor,
        "habitaciones": estate.get("rooms"),
        "banos": estate.get("bathrooms"),
        "planta": features.get("floor"),
        "anio_construccion": features.get("build_year"),
        "anio_reforma": features.get("renovation_year"),
        "categoria": features.get("category"),
        "ascensor": features.get("elevator"),
        "balcones": features.get("balconies"),
        "terrazas": features.get("terraces"),
        "jardin": features.get("garden"),
        "calefaccion": features.get("heating"),
        "clase_energetica": energy_data.get("class"),
        "points_of_interest": json.dumps(estate.get("points_of_interest")),
        "title": estate.get("title"),
        "description": estate.get("description"),
    }

    # Normalizar strings vacíos a None
    for campo in datos:
        if datos[campo] == "":
            datos[campo] = None

    return datos


In [ ]:
'''=================================================
FASE 2 (TECNOCASA) BLOQUE 0: Subida de CSVs previos
=================================================='''

# Sube URLs_recolectadas.csv obligatoriamente.
# Si ya tienes dataset_final.csv y urls_caidas.csv de ejecuciones anteriores, súbelos también.
# Si es la primera vez que corres la Fase 2, no los tendrás: no pasa nada, el código los crea.
subidos = files.upload()

print("Archivos subidos:", list(subidos.keys()))

Saving dataset_final.csv to dataset_final.csv
Saving urls_caidas.csv to urls_caidas.csv
Saving URLs_recolectadas.csv to URLs_recolectadas.csv
Archivos subidos: ['dataset_final.csv', 'urls_caidas.csv', 'URLs_recolectadas.csv']


In [ ]:
'''==============================================================
FASE 2 (TECNOCASA) BLOQUE 1: Calcular URLs pendientes de procesar
==============================================================='''

# Cargar todas las URLs recolectadas en Fase 1
todas_las_urls = set()
with open("URLs_recolectadas.csv", newline="", encoding="utf-8-sig") as f:
    lector = csv.DictReader(f)
    for fila in lector:
        todas_las_urls.add(fila["url_detalle"])

print(f"Total de URLs recolectadas: {len(todas_las_urls)}")

# Cargar URLs ya procesadas con éxito (si el archivo existe de una ejecución anterior)
urls_ya_procesadas = set()
if os.path.exists("dataset_final.csv"):
    with open("dataset_final.csv", newline="", encoding="utf-8-sig") as f:
        lector = csv.DictReader(f, delimiter=";")
        for fila in lector:
            urls_ya_procesadas.add(fila["detail_url"])

print(f"URLs ya procesadas con éxito: {len(urls_ya_procesadas)}")

# Cargar URLs que ya sabemos que fallaron (para no reintentarlas)
urls_caidas_previas = set()
if os.path.exists("urls_caidas.csv"):
    with open("urls_caidas.csv", newline="", encoding="utf-8-sig") as f:
        lector = csv.DictReader(f, delimiter=";")
        for fila in lector:
            urls_caidas_previas.add(fila["url"])

print(f"URLs marcadas como caídas previamente: {len(urls_caidas_previas)}")

# URLs que quedan por procesar en esta ejecución
urls_pendientes = todas_las_urls - urls_ya_procesadas - urls_caidas_previas

print(f"URLs pendientes de extraer en esta ejecución: {len(urls_pendientes)}")

Total de URLs recolectadas: 992
URLs ya procesadas con éxito: 979
URLs marcadas como caídas previamente: 3
URLs pendientes de extraer en esta ejecución: 10


In [ ]:
'''=================================================================
FASE 2 (TECNOCASA) BLOQUE 2: Bucle principal con guardado progresivo
=================================================================='''

# Nombres de los 21 campos que devuelve extraer_datos_inmueble()
campos = ["id", "detail_url", "tipo", "distrito", "barrio", "precio", "m2",
          "habitaciones", "banos", "planta", "anio_construccion", "anio_reforma",
          "categoria", "ascensor", "balcones", "terrazas", "jardin",
          "calefaccion", "clase_energetica", "points_of_interest",
          "title", "description"]

# Si dataset_final.csv no existe todavía, hay que escribir la cabecera primero
escribir_cabecera_dataset = not os.path.exists("dataset_final.csv")
escribir_cabecera_caidas = not os.path.exists("urls_caidas.csv")

contador_ok = 0
contador_fallo = 0

# Abrimos los dos CSVs en modo "a" (append): cada ficha se escribe al momento, no al final
# delimiter=";" para que Excel en español lo abra bien sin descuadrar columnas
with open("dataset_final.csv", "a", newline="", encoding="utf-8-sig") as f_dataset, \
     open("urls_caidas.csv", "a", newline="", encoding="utf-8-sig") as f_caidas:

    escritor_dataset = csv.DictWriter(f_dataset, fieldnames=campos, delimiter=";")
    escritor_caidas = csv.writer(f_caidas, delimiter=";")

    if escribir_cabecera_dataset:
        escritor_dataset.writeheader()
    if escribir_cabecera_caidas:
        escritor_caidas.writerow(["url"])

    for i, url in enumerate(urls_pendientes, start=1):
        datos = extraer_datos_inmueble(url)

        if datos is not None:
            escritor_dataset.writerow(datos)
            contador_ok += 1
        else:
            escritor_caidas.writerow([url])
            contador_fallo += 1

        # Forzamos escritura a disco cada ficha, para no perder nada si Colab se cae
        f_dataset.flush()
        f_caidas.flush()

        if i % 25 == 0:
            print(f"Progreso: {i}/{len(urls_pendientes)} — OK: {contador_ok} — Fallos: {contador_fallo}")

        time.sleep(1.5)  # pausa ética entre peticiones

print(f"\nBucle terminado. Nuevas fichas extraídas: {contador_ok}. Nuevos fallos: {contador_fallo}")


Bucle terminado. Nuevas fichas extraídas: 10. Nuevos fallos: 0


In [ ]:
'''==================================
FASE 2 (TECNOCASA) BLOQUE 3: Resumen
=================================='''

total_dataset = sum(1 for _ in open("dataset_final.csv", encoding="utf-8-sig")) - 1  # -1 por la cabecera
total_caidas = sum(1 for _ in open("urls_caidas.csv", encoding="utf-8-sig")) - 1

print(f"Fichas nuevas en esta ejecución: {contador_ok}")
print(f"URLs caídas nuevas en esta ejecución: {contador_fallo}")
print(f"Total acumulado en dataset_final.csv: {total_dataset}")
print(f"Total acumulado en urls_caidas.csv: {total_caidas}")


# Función de diagnóstico: repite la lógica de extraer_datos_inmueble()
# pero en vez de devolver None, indica el motivo exacto del fallo
def diagnosticar_fallo(url):
    try:
        respuesta = requests.get(url, timeout=10)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException as e:
        return f"ERROR DE RED: {e}"

    soup = BeautifulSoup(respuesta.text, "html.parser")
    etiqueta = soup.find("estate-show-v2")

    if etiqueta is None:
        return "NO SE ENCONTRÓ <estate-show-v2> (¿tipo de anuncio distinto o página vendida con otra plantilla?)"

    estate_raw = etiqueta.get(":estate")
    if estate_raw is None:
        return "LA ETIQUETA EXISTE PERO NO TIENE ATRIBUTO :estate"

    return "Debería haber funcionado (revisa manualmente)"


print("\nURLs caídas y motivo del fallo:")
with open("urls_caidas.csv", newline="", encoding="utf-8-sig") as f:
    lector = csv.DictReader(f, delimiter=";")
    for fila in lector:
        motivo = diagnosticar_fallo(fila["url"])
        print(f"{fila['url']}\n  → {motivo}\n")
        time.sleep(1.5)

Fichas nuevas en esta ejecución: 10
URLs caídas nuevas en esta ejecución: 0
Total acumulado en dataset_final.csv: 989
Total acumulado en urls_caidas.csv: 3

URLs caídas y motivo del fallo:
https://www.tecnocasa.es/venta/local-comercial/madrid/madrid/585198.html
  → ERROR DE RED: 404 Client Error: Not Found for url: https://www.tecnocasa.es/venta/local-comercial/madrid/madrid/585198.html

https://www.tecnocasa.es/venta/piso/madrid/madrid/620645.html
  → ERROR DE RED: 404 Client Error: Not Found for url: https://www.tecnocasa.es/venta/piso/madrid/madrid/620645.html

https://www.tecnocasa.es/venta/piso/madrid/madrid/664142.html
  → ERROR DE RED: 404 Client Error: Not Found for url: https://www.tecnocasa.es/venta/piso/madrid/madrid/664142.html



In [ ]:
'''============================================================================
FASE 2 (TECNOCASA) BLOQUE 4: Descarga de dataset_final.csv para subir a GitHub
============================================================================'''

files.download("dataset_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
'''=========================================================================
FASE 2 (TECNOCASA) BLOQUE 5: Descarga de urls_caidas.csv para subir a GitHub
========================================================================='''

files.download("urls_caidas.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
'''=======================================================
FASE 1 (REDPISO) BLOQUE 0: Imports y configuración inicial
======================================================='''

# ATENCIÓN!!!!! Borrar los archivos de Colab

import requests
from bs4 import BeautifulSoup
import csv
import os
import time
from google.colab import files  # para subir/descargar archivos en Colab

# Cabecera para simular un navegador real (evita bloqueos básicos por user-agent vacío)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

BASE_URL = "https://www.redpiso.es/venta-viviendas/madrid/madrid"
NOMBRE_CSV = "urls_recolectadas_redpiso.csv"
MAX_PAGINAS_SEGURIDAD = 150   # tope de seguridad, muy por encima de las páginas reales
PAUSA_ENTRE_PETICIONES = 1.5  # segundos de espera entre peticiones (cortesía con el servidor)

print("Bloque 0 completado: configuración cargada.")

Bloque 0 completado: configuración cargada.


In [ ]:
'''=========================================================
FASE 1 (REDPISO) BLOQUE 1: Subir el CSV de la sesión anterior (si ya existe)
========================================================='''

# Colab no tiene memoria entre sesiones, así que si ya recolectaste URLs antes,
# aquí las volvemos a cargar. Si es la primera vez, simplemente pulsa "Cancelar"
# en el diálogo de subida y el script empezará desde cero.

urls_previas = set()

print("Si ya tienes un urls_recolectadas_redpiso.csv de una sesión anterior, súbelo ahora.")
print("Si es la primera vez, pulsa 'Cancel upload' / cierra el diálogo.")

try:
    subido = files.upload()
    if NOMBRE_CSV in subido:
        with open(NOMBRE_CSV, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f, delimiter=";")
            for fila in reader:
                urls_previas.add(fila["url_detalle"])
        print(f"CSV cargado: {len(urls_previas)} URLs previas encontradas.")
    else:
        print("No se subió el archivo esperado. Empezamos desde cero.")
except Exception as e:
    print(f"No se subió ningún archivo ({e}). Empezamos desde cero.")

print(f"\nTotal de URLs previas cargadas: {len(urls_previas)}")

Si ya tienes un urls_recolectadas_redpiso.csv de una sesión anterior, súbelo ahora.
Si es la primera vez, pulsa 'Cancel upload' / cierra el diálogo.


Saving urls_recolectadas_redpiso.csv to urls_recolectadas_redpiso.csv
CSV cargado: 778 URLs previas encontradas.

Total de URLs previas cargadas: 778


In [ ]:
'''===========================================================================
FASE 1 (REDPISO) BLOQUE 2: Recorrer todas las páginas de Redpiso y extraer las URLs de anuncio
==========================================================================='''

# Patrón de paginación confirmado:
#   Página 1 -> BASE_URL
#   Página N -> BASE_URL/pagina-N   (N >= 2)
# Cuando una página no tiene anuncios (0 encontrados), hemos llegado al final real.

def url_pagina(n):
    if n == 1:
        return BASE_URL
    return f"{BASE_URL}/pagina-{n}"


def extraer_urls_inmueble(html):
    soup = BeautifulSoup(html, "html.parser")
    urls = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/inmueble/" in href:
            if href.startswith("/"):
                href = "https://www.redpiso.es" + href
            urls.add(href)
    return urls


urls_actuales = set()
pagina = 1

while pagina <= MAX_PAGINAS_SEGURIDAD:
    url = url_pagina(pagina)
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
    except Exception as e:
        print(f"Página {pagina} -> ERROR de conexión: {e}. Reintentando en 5s...")
        time.sleep(5)
        continue

    if r.status_code != 200:
        print(f"Página {pagina} -> status {r.status_code}. Detenemos aquí.")
        break

    urls_pagina = extraer_urls_inmueble(r.text)

    if len(urls_pagina) == 0:
        print(f"Página {pagina} -> 0 anuncios encontrados. Fin de la paginación real.")
        break

    urls_actuales |= urls_pagina
    print(f"Página {pagina} -> {len(urls_pagina)} anuncios | total acumulado: {len(urls_actuales)}")

    pagina += 1
    time.sleep(PAUSA_ENTRE_PETICIONES)

print(f"\nBloque 2 completado. URLs actuales encontradas en la web: {len(urls_actuales)}")

Página 1 -> 12 anuncios | total acumulado: 12
Página 2 -> 12 anuncios | total acumulado: 24
Página 3 -> 12 anuncios | total acumulado: 36
Página 4 -> 12 anuncios | total acumulado: 48
Página 5 -> 12 anuncios | total acumulado: 60
Página 6 -> 12 anuncios | total acumulado: 72
Página 7 -> 12 anuncios | total acumulado: 84
Página 8 -> 12 anuncios | total acumulado: 96
Página 9 -> 12 anuncios | total acumulado: 108
Página 10 -> 12 anuncios | total acumulado: 120
Página 11 -> 12 anuncios | total acumulado: 132
Página 12 -> 12 anuncios | total acumulado: 144
Página 13 -> 12 anuncios | total acumulado: 156
Página 14 -> 12 anuncios | total acumulado: 168
Página 15 -> 12 anuncios | total acumulado: 180
Página 16 -> 12 anuncios | total acumulado: 192
Página 17 -> 12 anuncios | total acumulado: 204
Página 18 -> 12 anuncios | total acumulado: 216
Página 19 -> 12 anuncios | total acumulado: 228
Página 20 -> 12 anuncios | total acumulado: 240
Página 21 -> 12 anuncios | total acumulado: 252
Página 22

In [ ]:
'''=====================================================================
FASE 1 (REDPISO) BLOQUE 3: Comparar lo recolectado ahora con lo que ya teníamos guardado
====================================================================='''

# La resta de conjuntos (set - set) nos da solo las URLs que no existían antes.

urls_nuevas = urls_actuales - urls_previas

print(f"URLs nuevas detectadas en esta sesión: {len(urls_nuevas)}")
if urls_nuevas:
    print("Ejemplo de URL nueva:", list(urls_nuevas)[0])

URLs nuevas detectadas en esta sesión: 6
Ejemplo de URL nueva: https://www.redpiso.es/inmueble/piso-en-venta-en-calle-de-manola-y-rosario-san-andres-villaverde-madrid-RP472026155988


In [ ]:
'''============================================
FASE 1 (REDPISO) BLOQUE 4: Resumen numérico de la actualización
============================================'''

# (Recordatorio: nunca borramos URLs antiguas, aunque el anuncio ya no aparezca
# en la web -> las conservamos como observación histórica, como en Tecnocasa)

total_antes = len(urls_previas)
total_nuevas = len(urls_nuevas)
total_despues = total_antes + total_nuevas

print("=== RESUMEN DE LA ACTUALIZACIÓN ===")
print(f"URLs antes de esta sesión:   {total_antes}")
print(f"URLs nuevas encontradas:     {total_nuevas}")
print(f"URLs totales tras la unión:  {total_despues}")
print(f"(URLs vistas hoy en la web que ya conocíamos: {len(urls_actuales) - total_nuevas})")

=== RESUMEN DE LA ACTUALIZACIÓN ===
URLs antes de esta sesión:   778
URLs nuevas encontradas:     6
URLs totales tras la unión:  784
(URLs vistas hoy en la web que ya conocíamos: 776)


In [ ]:
'''==================================================================
FASE 1 (REDPISO) BLOQUE 5: Guardar el CSV final combinando URLs previas + URLs nuevas
=================================================================='''

# Formato: separador ';' y encoding 'utf-8-sig' (compatibilidad Excel español)

urls_finales = urls_previas | urls_nuevas  # unión de ambos conjuntos, sin duplicados

with open(NOMBRE_CSV, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow(["url_detalle"])
    for url in sorted(urls_finales):
        writer.writerow([url])

print(f"Bloque 5 completado: {NOMBRE_CSV} guardado con {len(urls_finales)} URLs totales.")

Bloque 5 completado: urls_recolectadas_redpiso.csv guardado con 784 URLs totales.


In [ ]:
'''===================================================
FASE 1 (REDPISO) BLOQUE 6: Descargar el CSV actualizado a tu ordenador
==================================================='''

# (Recuerda: descárgalo en su propia celda, sin descargas simultáneas,
# y no lo abras/guardes desde Excel para no cambiar el formato)

files.download(NOMBRE_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
'''========================================================
FASE 2 (REDPISO) Bloque 0: Imports y configuración inicial
========================================================'''

# ATENCIÓN!!!! Borrar todos los archivos de Colab

import requests
from bs4 import BeautifulSoup
import csv
import os
import re
import time
from google.colab import files

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

CSV_URLS = "urls_recolectadas_redpiso.csv"
CSV_DATASET = "dataset_final_redpiso.csv"
CSV_CAIDAS = "urls_caidas_redpiso.csv"

PAUSA_ENTRE_PETICIONES = 1.5  # segundos, cortesía con el servidor

# Columnas del dataset final, en orden (planta va justo después de banos)
CAMPOS = [
    "referencia", "url_detalle", "tipo", "zona", "precio",
    "m2_construidos", "m2_utiles", "dormitorios", "banos", "planta",
    "anio_construccion", "estado",
    "ascensor", "balcones", "terrazas", "zonas_verdes_jardin", "piscina",
    "aire_acondicionado", "calefaccion", "calificacion_energetica",
    "garaje", "trastero", "exterior", "orientacion", "descripcion",
    "avisos_raros"
]

# --- Valores conocidos (categóricos, evitan "contaminación" de texto libre) ---
ESTADOS_CONOCIDOS = [
    "Perfecto estado", "A reformar", "Para entrar a vivir", "Obra nueva",
    "Depende del precio", "A estrenar", "Buen Estado", "Buen estado",
    "Reformado", "Sin especificar"
]
CALEFACCION_CONOCIDA = [
    "Individual", "Central", "Gas natural", "Eléctrica", "Gasóleo",
    "Bomba de calor", "No tiene", "Sin especificar"
]
ORIENTACIONES_CONOCIDAS = [
    "Norte-Sur", "Este-Oeste", "Noreste", "Noroeste", "Sureste", "Suroeste",
    "Norte", "Sur", "Este", "Oeste", "Sin especificar"
]
AC_TIPOS_CONOCIDOS = ["Frío/calor", "Solo frío", "Solo calor", "No tiene", "Sin especificar"]
ETIQUETAS_COMERCIALES = ["Oportunidad", "Reservado", "Vendido", "Premium"]
TIPOS_MAP = {
    "piso": "Piso", "atico": "Ático", "casa": "Casa", "chalet": "Chalet",
    "duplex": "Dúplex", "estudio": "Estudio", "loft": "Loft", "apartamento": "Apartamento"
}

# Términos de planta a buscar en la descripción (orden no importa)
PLANTA_TERMINOS = [
    "sótano", "semisótano", "entreplanta", "entresuelo", "bajo", "baja",
    "primera", "primero", "segunda", "segundo", "tercera", "tercero",
    "cuarta", "cuarto", "quinta", "quinto", "sexta", "sexto",
    "séptima", "séptimo", "octava", "octavo", "novena", "noveno",
    "décima", "décimo", "ático"
]

# Palabras clave para detectar contradicciones ("RARO") en cada campo booleano
PALABRAS_CLAVE = {
    "ascensor": ["ascensor"],
    "exterior": ["exterior"],
    "trastero": ["trastero"],
    "zonas_verdes_jardin": ["zona verde", "zonas verdes", "jardín", "jardin"],
    "piscina": ["piscina"],
    "garaje": ["garaje", "parking", "plaza de aparcamiento", "plaza de garaje"],
    "aire_acondicionado": ["aire acondicionado", "climatizador"],
    "balcones": ["balcón", "balcones"],
}

print("Bloque 0 completado: configuración cargada.")

Bloque 0 completado: configuración cargada.


In [ ]:
'''==========================================================================
FASE 2 (REDPIDO) Bloque 1: Subir los CSVs de sesiones anteriores (si existen)
==========================================================================='''

print("Sube urls_recolectadas_redpiso.csv (obligatorio):")
subido = files.upload()

urls_recolectadas = []
if CSV_URLS in subido:
    with open(CSV_URLS, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter=";")
        for fila in reader:
            urls_recolectadas.append(fila["url_detalle"])
    print(f"URLs recolectadas cargadas: {len(urls_recolectadas)}")
else:
    raise Exception("No se subió urls_recolectadas_redpiso.csv. Este archivo es obligatorio para continuar.")

print("\nSi ya tienes dataset_final_redpiso.csv de una sesión anterior, súbelo ahora. Si no, cancela el diálogo.")
urls_procesadas = set()
try:
    subido2 = files.upload()
    if CSV_DATASET in subido2:
        with open(CSV_DATASET, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f, delimiter=";")
            for fila in reader:
                urls_procesadas.add(fila["url_detalle"])
        print(f"Dataset previo cargado: {len(urls_procesadas)} fichas ya extraídas.")
    else:
        print("No se subió dataset_final_redpiso.csv. Empezamos el dataset desde cero.")
except Exception:
    print("No se subió ningún dataset previo. Empezamos desde cero.")

print("\nSi ya tienes urls_caidas_redpiso.csv de una sesión anterior, súbelo ahora. Si no, cancela el diálogo.")
urls_caidas_previas = set()
try:
    subido3 = files.upload()
    if CSV_CAIDAS in subido3:
        with open(CSV_CAIDAS, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f, delimiter=";")
            for fila in reader:
                urls_caidas_previas.add(fila["url_detalle"])
        print(f"URLs caídas cargadas: {len(urls_caidas_previas)}")
    else:
        print("No se subió urls_caidas_redpiso.csv. Sin caídas registradas todavía.")
except Exception:
    print("No se subió ningún registro de caídas. Empezamos desde cero.")

Sube urls_recolectadas_redpiso.csv (obligatorio):


Saving urls_recolectadas_redpiso.csv to urls_recolectadas_redpiso.csv
URLs recolectadas cargadas: 784

Si ya tienes dataset_final_redpiso.csv de una sesión anterior, súbelo ahora. Si no, cancela el diálogo.


No se subió dataset_final_redpiso.csv. Empezamos el dataset desde cero.

Si ya tienes urls_caidas_redpiso.csv de una sesión anterior, súbelo ahora. Si no, cancela el diálogo.


No se subió urls_caidas_redpiso.csv. Sin caídas registradas todavía.


In [ ]:
'''=============================================================================
FASE 2 (REDPISO) Bloque 2: Calcular qué URLs quedan por procesar en esta sesión
============================================================================='''

set_recolectadas = set(urls_recolectadas)
urls_pendientes = set_recolectadas - urls_procesadas - urls_caidas_previas

print("=== ESTADO ANTES DE ESTA EJECUCIÓN ===")
print(f"Total URLs recolectadas:        {len(set_recolectadas)}")
print(f"Ya procesadas con éxito:        {len(urls_procesadas)}")
print(f"Marcadas como caídas (previas): {len(urls_caidas_previas)}")
print(f"Pendientes de extraer ahora:    {len(urls_pendientes)}")

=== ESTADO ANTES DE ESTA EJECUCIÓN ===
Total URLs recolectadas:        784
Ya procesadas con éxito:        0
Marcadas como caídas (previas): 0
Pendientes de extraer ahora:    784


In [ ]:
'''=================================================================
FASE 2 (REDPISO) Bloque 3: Extracción de datos + guardado progresivo
=================================================================='''

# LÓGICA DE INFERENCIA "No" + DETECCIÓN "RARO" (acordada):
# Para los campos booleanos confirmados como estructurados (ascensor, exterior,
# trastero, zonas_verdes_jardin, piscina, garaje, aire_acondicionado):
#   - Si el campo aparece estructurado en la web -> se usa ese valor tal cual.
#   - Si NO aparece -> por defecto es "No" (Redpiso omite la etiqueta cuando no aplica).
#   - EXCEPCIÓN: si la palabra clave del campo aparece mencionada en la descripción
#     sin una negación cercana ("sin ascensor", "no tiene trastero"...), no se asume
#     "No" -> se marca como "RARO" para revisión manual, y se anota en 'avisos_raros'.

def limpiar_texto(html):
    soup = BeautifulSoup(html, "html.parser")
    texto = soup.get_text(separator=" ", strip=True)
    texto = re.sub(r"\s+", " ", texto)
    return texto, soup


def buscar(patron, texto, grupo=1, flags=0):
    m = re.search(patron, texto, flags)
    return m.group(grupo).strip() if m else None


def buscar_valor_conocido(valores_posibles, texto, etiqueta_regex):
    for valor in valores_posibles:
        patron = re.escape(valor) + r"\s*" + etiqueta_regex
        if re.search(patron, texto, re.IGNORECASE):
            return valor
    return None


def inferir_booleano(valor_estructurado, campo, descripcion):
    """Devuelve (valor, es_raro). valor_estructurado ya viene de un regex directo (Sí/No/None)."""
    if valor_estructurado in ("Sí", "No"):
        return valor_estructurado, False
    if not descripcion:
        return "No", False
    for palabra in PALABRAS_CLAVE.get(campo, []):
        patron_negacion = rf"\b(sin|no tiene|no dispone de|no cuenta con|carece de)\s+(el\s+|la\s+|un\s+|una\s+)?{re.escape(palabra)}"
        if re.search(patron_negacion, descripcion, re.IGNORECASE):
            return "No", False
        if re.search(rf"\b{re.escape(palabra)}", descripcion, re.IGNORECASE):
            return "RARO", True
    return "No", False


def inferir_numero(valor_estructurado, campo, descripcion, default="0"):
    if valor_estructurado is not None:
        return valor_estructurado, False
    if not descripcion:
        return default, False
    for palabra in PALABRAS_CLAVE.get(campo, []):
        if re.search(rf"\b{re.escape(palabra)}", descripcion, re.IGNORECASE):
            return "RARO", True
    return default, False


def extraer_planta(descripcion):
    if not descripcion:
        return None
    patron_terminos = "|".join(PLANTA_TERMINOS)
    m = re.search(rf"\b({patron_terminos})\s+planta\b", descripcion, re.IGNORECASE)
    if m:
        return m.group(1).capitalize()
    m = re.search(rf"\bplanta\s+({patron_terminos})\b", descripcion, re.IGNORECASE)
    if m:
        return m.group(1).capitalize()
    m = re.search(r"(\d+)\s*[ªº]\s*planta", descripcion, re.IGNORECASE)
    if m:
        return f"{m.group(1)}ª"
    m = re.search(r"planta\s*(\d+)", descripcion, re.IGNORECASE)
    if m:
        return f"{m.group(1)}ª"
    return None


def extraer_datos_inmueble(url):
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.raise_for_status()
    texto, soup = limpiar_texto(r.text)

    datos = {campo: None for campo in CAMPOS}
    raros = []

    datos["url_detalle"] = url
    datos["referencia"] = buscar(r"(RP[0-9A-Za-z]+)/?$", url)

    tipo_raw = buscar(r"/inmueble/([a-z]+)-en-venta", url)
    datos["tipo"] = TIPOS_MAP.get(tipo_raw, tipo_raw.capitalize() if tipo_raw else None)

    datos["zona"] = buscar(r"Viviendas en ([^,]+), Madrid", texto)
    datos["precio"] = buscar(r"([\d\.]{4,})\s*€", texto)
    datos["m2_construidos"] = buscar(r"([\d\.,]+)\s*m²\s*Construidos", texto)
    datos["m2_utiles"] = buscar(r"([\d\.,]+)\s*m²\s*Útiles", texto)
    datos["dormitorios"] = buscar(r"(\d+)\s*Dormitorios", texto)
    datos["banos"] = buscar(r"(\d+)\s*Baños", texto)
    datos["anio_construccion"] = buscar(r"(\d{4})\s*Año", texto)
    datos["calificacion_energetica"] = buscar(r"([A-G]|Sin especificar)\s*Cal\.\s*energ", texto)
    datos["estado"] = buscar_valor_conocido(ESTADOS_CONOCIDOS, texto, "Estado")
    datos["calefaccion"] = buscar_valor_conocido(CALEFACCION_CONOCIDA, texto, "Calefacción")
    datos["orientacion"] = buscar_valor_conocido(ORIENTACIONES_CONOCIDAS, texto, "Orientación")

    # Descripción real: entre "Galería de fotos" y "Cuota de comunidad:"
    idx_inicio = texto.find("Galería de fotos")
    idx_fin = texto.find("Cuota de comunidad:")
    descripcion = None
    if idx_inicio != -1 and idx_fin != -1 and idx_fin > idx_inicio:
        descripcion = texto[idx_inicio + len("Galería de fotos"):idx_fin].strip()
    datos["descripcion"] = descripcion

    datos["planta"] = extraer_planta(descripcion)

    # --- Campos booleanos con inferencia "No" + detección RARO ---
    ascensor_raw = buscar(r"(Sí|No)\s*Ascensor", texto)
    datos["ascensor"], raro = inferir_booleano(ascensor_raw, "ascensor", descripcion)
    if raro: raros.append("ascensor")

    exterior_raw = buscar(r"(Sí|No)\s*Exterior", texto)
    datos["exterior"], raro = inferir_booleano(exterior_raw, "exterior", descripcion)
    if raro: raros.append("exterior")

    trastero_raw = buscar(r"(Sí|No)\s*Trastero", texto)
    datos["trastero"], raro = inferir_booleano(trastero_raw, "trastero", descripcion)
    if raro: raros.append("trastero")

    zonas_raw = buscar(r"(Sí|No)\s*Zonas verdes", texto)
    datos["zonas_verdes_jardin"], raro = inferir_booleano(zonas_raw, "zonas_verdes_jardin", descripcion)
    if raro: raros.append("zonas_verdes_jardin")

    piscina_raw = buscar(r"(Sí|No)\s*Piscina", texto)
    datos["piscina"], raro = inferir_booleano(piscina_raw, "piscina", descripcion)
    if raro: raros.append("piscina")

    garaje_raw = buscar(r"(Sí|No|\d+)\s*(?:Garaje|Parking)", texto)
    if garaje_raw is not None:
        datos["garaje"] = garaje_raw
    else:
        datos["garaje"], raro = inferir_booleano(None, "garaje", descripcion)
        if raro: raros.append("garaje")

    # Aire acondicionado: se deriva del tipo de A/A ("No tiene" -> No, cualquier otro tipo -> Sí)
    ac_tipo = buscar_valor_conocido(AC_TIPOS_CONOCIDOS, texto, "A/A")
    if ac_tipo is not None:
        datos["aire_acondicionado"] = "No" if ac_tipo == "No tiene" else "Sí"
    else:
        datos["aire_acondicionado"], raro = inferir_booleano(None, "aire_acondicionado", descripcion)
        if raro: raros.append("aire_acondicionado")

    # --- Campos numéricos con inferencia 0 + detección RARO ---
    terrazas_raw = buscar(r"(\d+)\s*Terrazas", texto)
    datos["terrazas"], raro = inferir_numero(terrazas_raw, "balcones", descripcion)  # placeholder, se corrige abajo
    datos["terrazas"] = terrazas_raw if terrazas_raw is not None else "0"

    balcones_raw = buscar(r"(\d+)\s*Balcones", texto)
    datos["balcones"], raro = inferir_numero(balcones_raw, "balcones", descripcion)
    if raro: raros.append("balcones")

    # Etiqueta comercial (no forma parte del dataset final pedido, se omite del CSV
    # pero puede añadirse fácilmente si luego se quiere: buscar "Comparte X Financiación")

    datos["avisos_raros"] = ",".join(raros) if raros else None

    return datos


def cargar_ids_existentes(path_csv, columna="url_detalle"):
    if not os.path.exists(path_csv):
        return set()
    ids = set()
    with open(path_csv, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter=";")
        for fila in reader:
            ids.add(fila[columna])
    return ids


def guardar_fila_dataset(path_csv, fila_dict):
    existe = os.path.exists(path_csv)
    with open(path_csv, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=";")
        if not existe:
            writer.writeheader()
        writer.writerow(fila_dict)


def guardar_url_caida(path_csv, url):
    existe = os.path.exists(path_csv)
    with open(path_csv, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f, delimiter=";")
        if not existe:
            writer.writerow(["url_detalle"])
        writer.writerow([url])


nuevas_fichas = 0
nuevos_fallos = 0

for i, url in enumerate(sorted(urls_pendientes), start=1):
    try:
        datos = extraer_datos_inmueble(url)
        guardar_fila_dataset(CSV_DATASET, datos)
        nuevas_fichas += 1
        aviso = f" [RARO: {datos['avisos_raros']}]" if datos["avisos_raros"] else ""
        print(f"[{i}/{len(urls_pendientes)}] OK{aviso} -> {url}")
    except Exception as e:
        guardar_url_caida(CSV_CAIDAS, url)
        nuevos_fallos += 1
        print(f"[{i}/{len(urls_pendientes)}] FALLO ({e}) -> {url}")

    time.sleep(PAUSA_ENTRE_PETICIONES)

print(f"\nBloque 3 completado. Fichas nuevas: {nuevas_fichas} | Fallos nuevos: {nuevos_fallos}")

[1/784] OK -> https://www.redpiso.es/inmueble/RP252026154968
[2/784] OK [RARO: exterior] -> https://www.redpiso.es/inmueble/RP392026150813
[3/784] OK [RARO: zonas_verdes_jardin] -> https://www.redpiso.es/inmueble/apartamento-en-venta-en-calle-concepcion-bahamonde-fuente-del-berro-salamanca-madrid-madrid-RP1242024125418
[4/784] OK [RARO: garaje] -> https://www.redpiso.es/inmueble/apartamento-en-venta-en-calle-de-canillas-prosperidad-chamartin-madrid-RP1282026152639
[5/784] OK [RARO: aire_acondicionado] -> https://www.redpiso.es/inmueble/apartamento-en-venta-en-calle-de-francos-rodriguez-bellas-vistas-tetuan-madrid-RP1062026152095
[6/784] OK [RARO: exterior,trastero] -> https://www.redpiso.es/inmueble/apartamento-en-venta-en-calle-de-la-madera-universidad-centro-madrid-RP1292026152593
[7/784] OK [RARO: balcones] -> https://www.redpiso.es/inmueble/apartamento-en-venta-en-calle-de-la-sierra-del-castillo-san-diego-puente-de-vallecas-madrid-RP152026151614
[8/784] OK [RARO: zonas_verdes_jardi

In [ ]:
'''===============================================
FASE 2 (REDPISO) Bloque 4: Resumen de la ejecución
================================================'''

total_dataset_acumulado = len(cargar_ids_existentes(CSV_DATASET))
total_caidas_acumulado = len(cargar_ids_existentes(CSV_CAIDAS))

print("=== RESUMEN DE LA EJECUCIÓN ===")
print(f"Fichas nuevas extraídas en esta ejecución: {nuevas_fichas}")
print(f"URLs caídas nuevas en esta ejecución:      {nuevos_fallos}")
print(f"Total acumulado en dataset_final_redpiso:  {total_dataset_acumulado}")
print(f"Total acumulado en urls_caidas_redpiso:    {total_caidas_acumulado}")

=== RESUMEN DE LA EJECUCIÓN ===
Fichas nuevas extraídas en esta ejecución: 784
URLs caídas nuevas en esta ejecución:      0
Total acumulado en dataset_final_redpiso:  784
Total acumulado en urls_caidas_redpiso:    0


In [ ]:
'''========================================================
FASE 2 (REDPISO) Bloque 5: Descargar el dataset actualizado
========================================================'''

files.download(CSV_DATASET)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
'''========================================================
FASE 2 (REDPISO) Bloque 6: Descargar URLs caídas
========================================================'''

files.download(CSV_CAIDAS)

FileNotFoundError: Cannot find file: urls_caidas_redpiso.csv

prueba

In [2]:
# Bloque 0 — Inspección completa de un anuncio de Tecnocasa (JSON crudo)

import requests
import re
import json

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

# Pega aquí una URL real de un anuncio de venta de tu propio dataset (urls_recolectadas_tecnocasa.csv)
URL_EJEMPLO = "https://www.tecnocasa.es/venta/piso/madrid/madrid/661276.html"

resp = requests.get(URL_EJEMPLO, headers=HEADERS, timeout=15)
html = resp.text

# Tecnocasa mete el objeto completo del anuncio en el atributo :estate="{...}" del componente Vue
match = re.search(r':estate="([^"]+)"', html)

if not match:
    print("No se encontró el atributo :estate en esta URL. Comprueba que sea un anuncio individual válido.")
else:
    raw = match.group(1)
    # El HTML escapa comillas como &quot; dentro del atributo -> hay que desescapar antes de json.loads
    raw_unescaped = (raw
                      .replace("&quot;", '"')
                      .replace("&amp;", "&")
                      .replace("&#39;", "'")
                      .replace("&lt;", "<")
                      .replace("&gt;", ">"))
    try:
        estate = json.loads(raw_unescaped)
    except json.JSONDecodeError as e:
        print(f"Error parseando JSON: {e}")
        print("Primeros 500 caracteres del raw para depurar:")
        print(raw_unescaped[:500])
    else:
        print(f"Anuncio: {URL_EJEMPLO}")
        print(f"Total de claves de primer nivel: {len(estate.keys())}\n")

        # Volcado legible de TODAS las claves y valores (incluye anidados)
        print(json.dumps(estate, indent=2, ensure_ascii=False))

Anuncio: https://www.tecnocasa.es/venta/piso/madrid/madrid/661276.html
Total de claves de primer nivel: 63

{
  "ad_type": "estate",
  "id": 661276,
  "address": "C. Soto Hidalgo",
  "agency": {
    "aicat": "https://tecnocasa-cdn.medialabtc.it/tecnocasa-immagini/md066/register.jpg",
    "active": 1,
    "address": "Avda Cantabria, 39 Local 2 28042 Madrid (M)",
    "annunciPubblicati": null,
    "address_light": "AVENIDA CANTABRIA, 39 LOCAL 2",
    "city": {
      "capital": 0,
      "id": 5312,
      "title": "Madrid",
      "slug": "madrid"
    },
    "country": "es",
    "district": "Alameda de Osuna",
    "district_slug": "alameda_de_osuna",
    "distance": null,
    "email": "md066@tecnocasa.es",
    "id": "md066",
    "images": null,
    "latitude": 40.4574,
    "longitude": -3.58583,
    "mobile": "637.383.712",
    "name": "Estudio Alameda De Osuna Sl",
    "network": "tec",
    "opening_hours": "Lunes a Viernes: 09:30 a 14:00 - 17:00 a 21:00. Sábados: 10:00 a 14:00 y de 17:00 

In [4]:
# Bloque Upload — Subida del CSV de URLs

from google.colab import files
subido = files.upload()  # selecciona tu URLs_recolectadas.csv
print(list(subido.keys()))  # confirma el nombre exacto detectado

Saving URLs_recolectadas.csv to URLs_recolectadas (1).csv
['URLs_recolectadas (1).csv']


In [6]:
# Bloque 0b — Descubrimiento de esquema: todas las claves posibles en anuncios de Tecnocasa

import requests
import re
import json
import time
import csv
from collections import defaultdict

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

# --- Config ---
CSV_URLS = "URLs_recolectadas.csv"
TAMANO_MUESTRA = 40  # nº de anuncios a inspeccionar (ajusta si quieres más cobertura)
COLUMNA_URL = "url_detalle"  # ajusta si en tu CSV la columna se llama distinto


def obtener_estate(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    match = re.search(r':estate="([^"]+)"', resp.text)
    if not match:
        return None
    raw = (match.group(1)
           .replace("&quot;", '"').replace("&amp;", "&")
           .replace("&#39;", "'").replace("&lt;", "<").replace("&gt;", ">"))
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None


def recorrer_claves(obj, prefijo=""):
    """Genera todas las rutas de clave (dot-notation), aplanando dicts y mirando dentro de listas."""
    rutas = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            ruta = f"{prefijo}.{k}" if prefijo else k
            rutas.append((ruta, v))
            rutas.extend(recorrer_claves(v, ruta))
    elif isinstance(obj, list):
        if obj:
            # Solo miramos dentro del primer elemento como representativo de la estructura de la lista
            rutas.extend(recorrer_claves(obj[0], f"{prefijo}[]"))
    return rutas


# --- Leer muestra de URLs ---
with open(CSV_URLS, encoding="utf-8-sig") as f:
    reader = csv.DictReader(f, delimiter=";")
    print("Columnas detectadas:", reader.fieldnames)
    urls = [row[COLUMNA_URL] for row in reader]

muestra = urls[:TAMANO_MUESTRA]
print(f"Escaneando {len(muestra)} anuncios de {len(urls)} totales...")

# --- Escanear y acumular esquema ---
esquema = defaultdict(lambda: {"apariciones": 0, "vacios": 0, "ejemplo": None, "tipos": set()})

for i, url in enumerate(muestra, 1):
    estate = obtener_estate(url)
    if estate is None:
        print(f"[{i}/{len(muestra)}] fallo: {url}")
        continue

    for ruta, valor in recorrer_claves(estate):
        info = esquema[ruta]
        info["apariciones"] += 1
        info["tipos"].add(type(valor).__name__)
        vacio = valor is None or valor == "" or valor == [] or valor == {}
        if vacio:
            info["vacios"] += 1
        elif info["ejemplo"] is None:
            ejemplo_str = str(valor)
            info["ejemplo"] = ejemplo_str[:80] + ("..." if len(ejemplo_str) > 80 else "")

    print(f"[{i}/{len(muestra)}] OK ({url})")
    time.sleep(0.5)  # pacing ético

# --- Volcado del esquema ---
print(f"\n\n{'='*90}")
print(f"ESQUEMA DESCUBIERTO — {len(esquema)} rutas de clave únicas sobre {len(muestra)} anuncios")
print(f"{'='*90}\n")

filas_csv = []
for ruta in sorted(esquema.keys()):
    info = esquema[ruta]
    pct_relleno = 100 * (info["apariciones"] - info["vacios"]) / len(muestra)
    tipos = ",".join(info["tipos"])
    print(f"{ruta:55} | tipo={tipos:15} | relleno={pct_relleno:5.1f}% | ej: {info['ejemplo']}")
    filas_csv.append({
        "ruta": ruta,
        "tipo": tipos,
        "porcentaje_relleno": round(pct_relleno, 1),
        "ejemplo": info["ejemplo"],
    })

with open("esquema_tecnocasa.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["ruta", "tipo", "porcentaje_relleno", "ejemplo"], delimiter=";")
    writer.writeheader()
    writer.writerows(filas_csv)

print(f"\nGuardado en esquema_tecnocasa.csv — descárgalo con files.download('esquema_tecnocasa.csv')")


Columnas detectadas: ['url_detalle']
Escaneando 40 anuncios de 992 totales...
[1/40] OK (https://www.tecnocasa.es/venta/piso/madrid/madrid/666713.html)
[2/40] OK (https://www.tecnocasa.es/venta/piso/madrid/madrid/662232.html)
[3/40] OK (https://www.tecnocasa.es/venta/piso/madrid/madrid/661907.html)
[4/40] OK (https://www.tecnocasa.es/venta/local-comercial/madrid/madrid/662910.html)
[5/40] OK (https://www.tecnocasa.es/venta/local-comercial/madrid/madrid/665916.html)
[6/40] OK (https://www.tecnocasa.es/venta/piso/madrid/madrid/665849.html)
[7/40] OK (https://www.tecnocasa.es/venta/piso/madrid/madrid/657415.html)
[8/40] OK (https://www.tecnocasa.es/venta/piso/madrid/madrid/635754.html)
[9/40] OK (https://www.tecnocasa.es/venta/piso/madrid/madrid/652991.html)
[10/40] OK (https://www.tecnocasa.es/venta/piso/madrid/madrid/651713.html)
[11/40] OK (https://www.tecnocasa.es/venta/piso/madrid/madrid/663473.html)
[12/40] OK (https://www.tecnocasa.es/venta/local-comercial/madrid/madrid/657064.html

In [7]:
# Bloque 0c — Investigación de campos ambiguos: box, car_places, exposure_primary/secondary, building_state

import requests
import re
import json
import time
import csv

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

CSV_URLS = "URLs_recolectadas.csv"
COLUMNA_URL = "url_detalle"
TAMANO_MUESTRA = 150  # más amplia que la del 0b para tener más chance de encontrar valores rellenos

CAMPOS_A_INVESTIGAR = {
    "features.box": lambda e: (e.get("features") or {}).get("box"),
    "features.car_places": lambda e: (e.get("features") or {}).get("car_places"),
    "exposure_primary": lambda e: e.get("exposure_primary"),
    "exposure_secondary": lambda e: e.get("exposure_secondary"),
    "energy_data.building_state": lambda e: (e.get("energy_data") or {}).get("building_state"),
}


def obtener_estate(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    match = re.search(r':estate="([^"]+)"', resp.text)
    if not match:
        return None
    raw = (match.group(1)
           .replace("&quot;", '"').replace("&amp;", "&")
           .replace("&#39;", "'").replace("&lt;", "<").replace("&gt;", ">"))
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None


def limpiar_descripcion(html_desc):
    if not html_desc:
        return ""
    texto = re.sub(r'<[^>]+>', ' ', html_desc)
    texto = texto.replace('&nbsp;', ' ')
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto


with open(CSV_URLS, encoding="utf-8-sig") as f:
    reader = csv.DictReader(f, delimiter=";")
    urls = [row[COLUMNA_URL] for row in reader]

muestra = urls[:TAMANO_MUESTRA]
print(f"Escaneando {len(muestra)} anuncios buscando valores no vacíos en campos ambiguos...\n")

hallazgos = {campo: [] for campo in CAMPOS_A_INVESTIGAR}

for i, url in enumerate(muestra, 1):
    estate = obtener_estate(url)
    if estate is None:
        continue

    for campo, extractor in CAMPOS_A_INVESTIGAR.items():
        valor = extractor(estate)
        if valor is not None and str(valor).strip() != "":
            hallazgos[campo].append({
                "url": url,
                "valor": valor,
                "descripcion": limpiar_descripcion(estate.get("description"))[:400],
            })

    if i % 25 == 0:
        print(f"  ...{i}/{len(muestra)} anuncios revisados")
    time.sleep(0.4)

# --- Volcado de resultados ---
print(f"\n{'='*90}")
for campo, casos in hallazgos.items():
    print(f"\n{campo}: {len(casos)} anuncios con valor no vacío (de {len(muestra)} revisados)")
    print("-" * 90)
    for c in casos[:5]:  # mostramos hasta 5 ejemplos por campo
        print(f"URL: {c['url']}")
        print(f"Valor: {c['valor']}")
        print(f"Descripción (extracto): {c['descripcion']}")
        print()

Escaneando 150 anuncios buscando valores no vacíos en campos ambiguos...

  ...25/150 anuncios revisados
  ...50/150 anuncios revisados
  ...75/150 anuncios revisados
  ...100/150 anuncios revisados
  ...125/150 anuncios revisados
  ...150/150 anuncios revisados


features.box: 0 anuncios con valor no vacío (de 150 revisados)
------------------------------------------------------------------------------------------

features.car_places: 0 anuncios con valor no vacío (de 150 revisados)
------------------------------------------------------------------------------------------

exposure_primary: 0 anuncios con valor no vacío (de 150 revisados)
------------------------------------------------------------------------------------------

exposure_secondary: 0 anuncios con valor no vacío (de 150 revisados)
------------------------------------------------------------------------------------------

energy_data.building_state: 0 anuncios con valor no vacío (de 150 revisados)
---------------------

In [8]:
# Bloque Prueba — Extracción completa Tecnocasa v2 sobre 40 URLs aleatorias -> Excel

import requests
import re
import json
import time
import csv
import random
import pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

CSV_URLS = "URLs_recolectadas.csv"
COLUMNA_URL = "url_detalle"
TAMANO_MUESTRA = 40
SEMILLA = 42  # fija para poder reproducir la misma muestra si hace falta

VARIABLES_FINALES = [
    "id", "url_detalle", "tipo", "zona", "precio", "m2_construidos", "m2_utiles",
    "dormitorios", "banos", "planta", "anio_construccion", "ascensor", "balcones",
    "terrazas", "zonas_verdes_jardin", "calefaccion", "calificacion_energetica",
    "descripcion", "estado", "piscina", "aire_acondicionado", "garaje", "trastero",
    "exterior", "orientacion", "tiene_raro", "avisos_raros",
]

# --- Utilidades básicas ---

def obtener_estate(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    match = re.search(r':estate="([^"]+)"', resp.text)
    if not match:
        return None
    raw = (match.group(1)
           .replace("&quot;", '"').replace("&amp;", "&")
           .replace("&#39;", "'").replace("&lt;", "<").replace("&gt;", ">"))
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None


def limpiar_descripcion(html_desc):
    if not html_desc:
        return ""
    texto = re.sub(r'<[^>]+>', ' ', html_desc)
    texto = texto.replace('&nbsp;', ' ')
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto


def extraer_numero(texto):
    if texto is None:
        return None
    m = re.search(r'\d+', str(texto))
    return int(m.group()) if m else None


# --- Minería de descripción para booleanos (estilo Redpiso) ---

PATRONES_BOOLEANOS = {
    "ascensor": [r"ascensor"],
    "balcones": [r"balc[oó]n(es)?"],
    "terrazas": [r"terraza(s)?"],
    "zonas_verdes_jardin": [r"jard[ií]n(es)?", r"zonas? ajardinadas?", r"zonas? verdes?"],
    "aire_acondicionado": [r"aire acondicionado", r"climatizad\w*"],
    "piscina": [r"piscina"],
    "garaje": [r"garaje", r"plaza de (parking|garaje|aparcamiento)"],
    "trastero": [r"trastero"],
    "exterior": [r"\bexterior\b"],
}

NEGACIONES = r'\b(sin|no tiene|no dispone|no cuenta con|carece de|no incluye)\b'

# Patrones que indican condición/ambigüedad -> RARO en vez de Si/No directo
AMBIGUOS = [
    r"opcional", r"posibilidad de", r"consultar", r"a parte", r"aparte",
    r"coste adicional", r"con suplemento", r"no incluid[oa]", r"por separado",
    r"comunitari[oa]", r"si se desea", r"bajo presupuesto",
]


def detectar_booleano_en_texto(texto_lower, patrones):
    """Devuelve 'negado' / 'ambiguo' / 'mencionado' / None según lo que encuentre primero."""
    for patron in patrones:
        for m in re.finditer(patron, texto_lower):
            ventana_previa = texto_lower[max(0, m.start() - 40):m.start()]
            ventana_posterior = texto_lower[m.end():m.end() + 40]
            contexto = ventana_previa + " " + ventana_posterior

            if re.search(NEGACIONES, ventana_previa):
                return "negado"
            if any(re.search(amb, contexto) for amb in AMBIGUOS):
                return "ambiguo"
            return "mencionado"
    return None


def resolver_booleano(nombre, valor_estructurado, texto_desc_lower, lista_avisos):
    estructurado_presente = bool(valor_estructurado and str(valor_estructurado).strip())
    resultado_desc = detectar_booleano_en_texto(texto_desc_lower, PATRONES_BOOLEANOS[nombre])

    if resultado_desc == "ambiguo":
        lista_avisos.append(f"{nombre}: mención ambigua/condicionada en descripción")
        return "RARO"

    if estructurado_presente and resultado_desc == "negado":
        lista_avisos.append(f"{nombre}: campo estructurado presente pero descripción lo niega")
        return "RARO"

    if estructurado_presente or resultado_desc == "mencionado":
        return "Si"

    # Ni estructurado ni descripción lo mencionan, o descripción lo niega explícitamente -> No
    return "No"


# --- Extracción de "estado" (categórico, solo descripción) ---

PATRONES_ESTADO = {
    "A reformar": [r"a reformar", r"para reformar", r"necesita reforma", r"para actualizar"],
    "Reformado": [r"reformad[oa]", r"reforma integral (realizada|hecha)", r"totalmente renovad[oa]"],
    "Buen estado": [r"buen estado", r"muy buen estado", r"excelente estado", r"impecable"],
    "Actualización": [r"estado de actualizaci[oó]n"],
}


def resolver_estado(texto_desc_lower, lista_avisos):
    encontrados = set()
    for categoria, patrones in PATRONES_ESTADO.items():
        if any(re.search(p, texto_desc_lower) for p in patrones):
            encontrados.add(categoria)

    if len(encontrados) == 0:
        return "No_especificado"
    if len(encontrados) == 1:
        return encontrados.pop()

    lista_avisos.append(f"estado: menciones contradictorias -> {', '.join(sorted(encontrados))}")
    return "RARO"


# --- Extracción de "orientacion" (categórico, solo descripción) ---

PUNTOS_CARDINALES = {
    "Norte": r"\bnorte\b",
    "Sur": r"\bsur\b",
    "Este": r"\beste\b",
    "Oeste": r"\boeste\b",
    "Noreste": r"\bnoreste\b",
    "Noroeste": r"\bnoroeste\b",
    "Sureste": r"\bsureste\b",
    "Suroeste": r"\bsuroeste\b",
}


def resolver_orientacion(texto_desc_lower):
    encontrados = [nombre for nombre, patron in PUNTOS_CARDINALES.items() if re.search(patron, texto_desc_lower)]
    if not encontrados:
        return "No_especificado"
    return ", ".join(encontrados)


# --- Extracción completa de un anuncio ---

def extraer_variables(estate):
    avisos = []
    features = estate.get("features") or {}
    energy = estate.get("energy_data") or {}

    desc_limpia = limpiar_descripcion(estate.get("description"))
    desc_lower = desc_limpia.lower()

    fila = {
        "id": estate.get("id"),
        "url_detalle": estate.get("detail_url"),
        "tipo": (estate.get("type") or {}).get("title"),
        "zona": estate.get("quarter"),
        "precio": estate.get("numeric_price"),
        "m2_construidos": float(estate["numeric_surface"]) if estate.get("numeric_surface") else None,
        "m2_utiles": None,  # no existe como campo en Tecnocasa, se deja vacío siempre
        "dormitorios": extraer_numero(estate.get("rooms")) or features.get("bedrooms"),
        "banos": extraer_numero(estate.get("bathrooms")),
        "planta": features.get("floor") or None,
        "anio_construccion": features.get("build_year") or (estate.get("dates") or {}).get("build_year"),
        "calefaccion": features.get("heating") or None,
        "calificacion_energetica": energy.get("class"),
        "descripcion": desc_limpia,
    }

    fila["ascensor"] = resolver_booleano("ascensor", features.get("elevator"), desc_lower, avisos)
    fila["balcones"] = resolver_booleano("balcones", features.get("balconies"), desc_lower, avisos)
    fila["terrazas"] = resolver_booleano("terrazas", features.get("terraces"), desc_lower, avisos)
    fila["zonas_verdes_jardin"] = resolver_booleano("zonas_verdes_jardin", features.get("garden"), desc_lower, avisos)
    fila["aire_acondicionado"] = resolver_booleano("aire_acondicionado", features.get("air_conditioning"), desc_lower, avisos)
    fila["piscina"] = resolver_booleano("piscina", None, desc_lower, avisos)
    fila["garaje"] = resolver_booleano("garaje", None, desc_lower, avisos)
    fila["trastero"] = resolver_booleano("trastero", None, desc_lower, avisos)
    fila["exterior"] = resolver_booleano("exterior", None, desc_lower, avisos)

    fila["estado"] = resolver_estado(desc_lower, avisos)
    fila["orientacion"] = resolver_orientacion(desc_lower)

    fila["tiene_raro"] = "Si" if avisos else "No"
    fila["avisos_raros"] = "; ".join(avisos) if avisos else None

    return fila


# --- Ejecución: muestra aleatoria de 40 URLs ---

with open(CSV_URLS, encoding="utf-8-sig") as f:
    reader = csv.DictReader(f, delimiter=";")
    todas_urls = [row[COLUMNA_URL] for row in reader]

random.seed(SEMILLA)
muestra = random.sample(todas_urls, min(TAMANO_MUESTRA, len(todas_urls)))

print(f"Procesando {len(muestra)} anuncios aleatorios de {len(todas_urls)} totales...\n")

filas = []
fallos = []

for i, url in enumerate(muestra, 1):
    estate = obtener_estate(url)
    if estate is None:
        print(f"[{i}/{len(muestra)}] FALLO: {url}")
        fallos.append(url)
        continue
    fila = extraer_variables(estate)
    filas.append(fila)
    marca = " ⚠️ RARO" if fila["tiene_raro"] == "Si" else ""
    print(f"[{i}/{len(muestra)}] OK — id {fila['id']}{marca}")
    time.sleep(0.5)  # pacing ético

print(f"\nCompletado: {len(filas)} anuncios extraídos, {len(fallos)} fallos.")

# --- Guardar como Excel ---

df = pd.DataFrame(filas, columns=VARIABLES_FINALES)
nombre_archivo = "prueba_tecnocasa_40.xlsx"
df.to_excel(nombre_archivo, index=False, engine="openpyxl")

print(f"\nGuardado: {nombre_archivo}")
print(f"Anuncios con algún RARO: {(df['tiene_raro'] == 'Si').sum()} de {len(df)}")

from google.colab import files
files.download(nombre_archivo)

Procesando 40 anuncios aleatorios de 992 totales...

[1/40] OK — id None
[2/40] OK — id 664420 ⚠️ RARO
[3/40] OK — id 663616
[4/40] OK — id 638135
[5/40] OK — id 662267
[6/40] OK — id 577813
[7/40] OK — id 665415
[8/40] OK — id 660715
[9/40] OK — id 666818
[10/40] OK — id None
[11/40] OK — id 645148
[12/40] OK — id 661077
[13/40] OK — id None
[14/40] OK — id 661176
[15/40] OK — id 658638
[16/40] OK — id 612718
[17/40] OK — id 656811
[18/40] OK — id None
[19/40] OK — id 652175
[20/40] OK — id 664592
[21/40] OK — id 634381
[22/40] OK — id 658699
[23/40] OK — id 575184
[24/40] OK — id 666414
[25/40] OK — id 662644
[26/40] OK — id 648401
[27/40] OK — id 656290
[28/40] OK — id 657086
[29/40] OK — id None
[30/40] OK — id 653320
[31/40] OK — id 625403
[32/40] OK — id 661796
[33/40] OK — id None
[34/40] OK — id 660105
[35/40] OK — id 665584
[36/40] OK — id 661512
[37/40] OK — id 637717
[38/40] OK — id 657415
[39/40] OK — id 651378
[40/40] OK — id 656373

Completado: 40 anuncios extraídos, 0 fa

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>